<h1 style="text-align:center; color: #15ab76;">
 Speech Emotion Recognition
</h1>

<h3 style="text-align:center; color: #189681">
Machine learning - Proyecto final
</h3>

<p style="text-align:center; color: #db663b"> 
Primer Semestre 2026<br>
Juana Bertone y Julia Pantin
</p>


### Parte 0 - Definicion
Que es SER?
Que es RAVDESS?
Que modalidades usamos?
Que emociones usamos?
Cual es la tarea?


SER significa "Speech Emotion Recognition", es una tarea de machine learning en la que mediante un audio de voz, queremos predecir la emocion expresada por el actor.
RAVDESS es la base de datos utilizada que contiene grabaciones de 24 actores (12 hombres y 12 mujeres) en distintos formatos(solo audio, audio y video, solo video). En este caso, usamos unicamente los audios, que cuentan con muestras de habla y canto.

Los archivos de RAVDESS tienen el formato MM-VV-EE-II-SS-RR-AA.wav.
MM (modality) que puede ser 01 (Audio-Video), 02 (Video-Only) o 03 (Audio-Only), por lo que todos nuestros archivos van a empezar con 03. CC (Channel) especifica si la muestra corresponde a habla (01 = speech) o canto (02 = song). EE (Emotion) representa la emoción expresada, en este caso usamos 8 que serian neutral, calma, felicidad, tristeza, enojo, miedo, sorpresa y disgusto. II (Intensity) indica la intensidad emocional (01 = normal, 02 = strong). SS (Statement) identifica la frase pronunciada (hay dos opciones), RR (Repetition) indica si se trata de la primera o segunda repetición de la grabación y por ultimo, AA (Actor) identifica al actor que realizó la interpretación (1-24). Los impares correponden a hombres y los pares mujeres.

Es un problema de clasificacion multiclase, y la tarea es poder predecir la emocion expresada mediante el audio.


desp redactar un poco mejor


In [42]:
modality_map = {
    "01": "audio_video",
    "02": "video_only",
    "03": "audio_only"
}#igual este no hace falta porque son todos 03

channel_map = {
    "01": "speech",
    "02": "song"
}

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

intensity_map = {
    "01": "normal",
    "02": "strong"
}

statement_map = {
    "01": "Kids are talking by the door",
    "02": "Dogs are sitting by the door"
}

repetition_map = {
    "01": 1,
    "02": 2
}


### Parte 1 - Construccion de metadata
A partir de los archivos tengo que obtener : file, actor, gender, emotion, intensity, statement, repetition, modality.

-> metadata_df

In [ ]:
import os
import pandas as pd

records = []
folders = [
    "../raw/Audio_Speech_Actors_01-24",
    "../raw/Audio_Song_Actors_01-24"
]
for root_folder in folders:
    for actor_folder in os.listdir(root_folder):
        actor_path = os.path.join(root_folder, actor_folder)
        if not os.path.isdir(actor_path):
            continue
        for filename in os.listdir(actor_path):
            if not filename.endswith(".wav"):
                continue
            file_path = os.path.join(actor_path, filename)
            name = filename.replace(".wav", "")
            modality, channel, emotion, intensity, statement, repetition, actor = name.split("-")
            actor = int(actor)
            gender = "female" if actor % 2 == 0 else "male"
            records.append({
                # file, actor, gender, emotion, intensity, statement, repetition, modality
                "file": file_path,
                "actor": actor,
                "gender": gender,
                "emotion": emotion_map[emotion],
                "intensity": intensity_map[intensity],
                "statement": statement_map[statement],
                "repetition": int(repetition),
                #"modality": modality_map[modality],
                "channel": channel_map[channel]
            })

metadata_df = pd.DataFrame(records)

print("Cantidad de muestras:", len(metadata_df))
metadata_df.head()



Cantidad de muestras: 2452


,file,actor,gender,emotion,intensity,statement,repetition,channel
0,../raw/Audio_Speech_Actors_01-24\Actor_01\03-0...,1,male,neutral,normal,Kids are talking by the door,1,speech
1,../raw/Audio_Speech_Actors_01-24\Actor_01\03-0...,1,male,neutral,normal,Kids are talking by the door,2,speech
2,../raw/Audio_Speech_Actors_01-24\Actor_01\03-0...,1,male,neutral,normal,Dogs are sitting by the door,1,speech
3,../raw/Audio_Speech_Actors_01-24\Actor_01\03-0...,1,male,neutral,normal,Dogs are sitting by the door,2,speech
4,../raw/Audio_Speech_Actors_01-24\Actor_01\03-0...,1,male,calm,normal,Kids are talking by the door,1,speech


In [44]:

#metadata_dt tiene solamente información del nombre del archivo, no del contenido del audio
# eso arranca en el 3 q hay q ver como hacerlo
print(metadata_df.shape)

(2452, 8)


### Parte 2 - EDA

2.1 Cantidad de muestras : speech, song, total

2.2 Emociones : distribucion de clases.

2.3 Intensidad : Normal vs Strong.

2.4 Genero : Male vs female.

2.5 Actores : Cantidad por actor.

2.6 Texto : Statement 1 vs Statement 2.

2.7 Cruces : Emocion x genero, Emocion x intensidad, Emocion x modalidad

2.8 Posibles fuentes de leakage : que variables podrian generar leakage. 

In [45]:
#desp analizarrrrrrrrrr todo 
#duda con lo de modality q hago esta bien sacarlo xq es todo 03, seria channel no

#cantidad de muestras (2.1)
metadata_df["channel"].value_counts()


channel
speech    1440
song      1012
Name: count, dtype: int64

In [46]:
# 2.2
metadata_df["emotion"].value_counts()

emotion
calm         376
happy        376
sad          376
angry        376
fearful      376
disgust      192
surprised    192
neutral      188
Name: count, dtype: int64

In [47]:
# 2.3
metadata_df["intensity"].value_counts()

intensity
normal    1320
strong    1132
Name: count, dtype: int64

In [48]:
#2.4
metadata_df["gender"].value_counts()

gender
male      1248
female    1204
Name: count, dtype: int64

In [49]:
#2.5
metadata_df["actor"].value_counts()

actor
1     104
2     104
3     104
4     104
5     104
6     104
7     104
8     104
9     104
10    104
11    104
12    104
13    104
14    104
15    104
16    104
17    104
19    104
20    104
21    104
23    104
22    104
24    104
18     60
Name: count, dtype: int64

In [50]:
#2.6
metadata_df["statement"].value_counts()

statement
Kids are talking by the door    1226
Dogs are sitting by the door    1226
Name: count, dtype: int64

In [51]:
#2.7
pd.crosstab(
    metadata_df["emotion"],
    metadata_df["gender"]
)


gender,female,male
emotion,,
angry,184,192
calm,184,192
disgust,96,96
fearful,184,192
happy,184,192
neutral,92,96
sad,184,192
surprised,96,96


In [52]:
pd.crosstab(
    metadata_df["emotion"],
    metadata_df["intensity"]
)

intensity,normal,strong
emotion,,
angry,188,188
calm,188,188
disgust,96,96
fearful,188,188
happy,188,188
neutral,188,0
sad,188,188
surprised,96,96


In [53]:
pd.crosstab(
    metadata_df["emotion"],
    metadata_df["channel"]
)

channel,song,speech
emotion,,
angry,184,192
calm,184,192
disgust,0,192
fearful,184,192
happy,184,192
neutral,92,96
sad,184,192
surprised,0,192


HACER 2.8
EXPLICAR LOS OTROS

### Parte 3 - Diseño del split

Train:
Actores ...

Validation:
Actores ...

Test:
Actores ...

#### A PARTIR DE ACA:

NO MÁS LIMPIEZA

NO MÁS EDA

NO MÁS MODIFICAR SPLITS

### Parte 4 - Feature extraction

Aca aparece : MFCC, Chroma, Spectral Contrast, ZCR y RMS.

Guardar : X_train, X_val, X_test

### Parte 5 - Baseline

Logistic Regression

### Parte 6 - Pipeline 

Definir :

Extracción features

↓

Normalización

↓

Modelo

↓

Evaluación

### Parte 7 - Experimentos

Experimento A : Speech

Experimento B : Song

Experimento C : Speech + Song

Experimento D : Comparacion 

Experimento que aun no sabemos si vas a hacer: Speech -> Song y Song -> Speech

### Parte 8 - Resultados
Matrices de confusion.

Accuracy.

F1.

Analisis de errores.

Emociones mas dificiles.

Emociones mas faciles.

### Parte 9 - Conclusiones